## Data Distribution & Consistency
Hasta este punto, ya sabemos si faltan datos o existen duplicados; aohra queremos conocer qué valores contienen las variables numéricas, cómo se distribuyen y si encontramos cantidades que ameriten investigación.

**Objetivo**. Identifciar rangos, valores extremos y posibles inconsistencias sin corregirlos todavía.



In [0]:
import pandas as pd
from pathlib import Path # Para trabajar con rutas de archivos de manera más organizada


# Contrucción de rutas de datasets y validación de archivos exists
DATA_PATH = Path('../data/raw')

# Lectura de dataset
sales = DATA_PATH / 'Sales_dataset.csv'
print("Sales file exists:", sales.exists())

inventory = DATA_PATH / 'Inventory_dataset.csv'
print("Inventory file exists:", inventory.exists())

# Carga de datos
dfSales = pd.read_csv(sales, dtype={"cd_NumParte": "string"})

dfInventory = pd.read_csv(inventory, dtype= {"cd_NumParte": "string"})

In [0]:
# Identifciar columnas numéricas
sales_numeric_columns = dfSales.select_dtypes(
    include ="number"
).columns.tolist()

inventory_numeric_columns = dfInventory.select_dtypes(
    include ="number"
).columns.tolist()

print("Columnas numéricas de Sales")
print(sales_numeric_columns)

print("\nColumnas numéricas de Inventory")
print(inventory_numeric_columns)

In [0]:
#Estadística descriptivade Sales

display(
    dfSales[sales_numeric_columns].describe() #.T, transpone el resultado
)

In [0]:
#Estadística descriptiva de Inventory

display(
    dfInventory[inventory_numeric_columns].describe().T #T, transpone el resultado
)

In [0]:
# Seleccionar variables relevantes, no todas las columnas numéricas nos interesan igual.
sales_analysis_columns = [
    "ct_Cantidad",
    "im_PrecioVenta"
]

display(
    dfSales[sales_analysis_columns].describe().T
)

# Para inventario, primero revisar los nombres reales
print(dfInventory.columns.to_list())

inventory_analysis_columns = [
    "ct_Existencia",
    "ct_Reservado",
    "nu_PuntoReorden",
    "im_ValorInventario",
    "ct_VentasUltimos12Meses",
    "ct_DiasEnInventario"
]

display(
    dfInventory[inventory_analysis_columns].describe().T)

In [0]:
#Cantidades negativas, cero y positivas
negative_sales = dfSales[
    dfSales["ct_Cantidad"] < 0
]

zero_sales = dfSales[
    dfSales["ct_Cantidad"] == 0
]

positive_sales = dfSales[
    dfSales["ct_Cantidad"] > 0
]
# Cantidad de ventas negativas, cero y positivas
print("Ventas negativas:", len(negative_sales))
print("Ventas en cero:", len(zero_sales))
print("Ventas positivas:", len(positive_sales))

con el análisis anterior tenemos 3 categorías:
- < 0 -> investigar
- = 0 -> investigar
- > 0 -> comportamiento esperado inicialmente

No concluimos todavía que negativo = incorrecto, porque una cantidad egativa puede representar una devolución, una cancelación o algún movimiento permitido por el sistema.

Primero conocemos la regla de negocio.

In [0]:
# Validar precios
invalid_prices = dfSales[
    dfSales["im_PrecioVenta"] <= 0
]

print("Precios inválidos:", len(invalid_prices))

#display(invalid_prices.head(10)) #Empty dataset

##Pregunta de Negocio
¿Puede existir legítimamente una refacción con precio de venta igual a cero?, esto genera un registro para revisión.

##IQR: Interquartile Range

In [0]:
#Detectar posibles outliers
#IQR: Interquartile Range

Q1 = dfSales["ct_Cantidad"].quantile(0.25)
Q3 = dfSales["ct_Cantidad"].quantile(0.75)
IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Límite inferior:", lower_limit)
print("Límite superior:", upper_limit)

La lógica es:
**IQR = Q3 - Q1**

y usamos como regla exploratoria:
Límite inferior = Q1 - 1.5(IQR)
Límite superior = Q3 + 1.5(IQR)

In [0]:
#Ver los posibles outliers
quantity_outliers = dfSales [
    (dfSales["ct_Cantidad"] < lower_limit) | 
    (dfSales["ct_Cantidad"] > upper_limit)
]

print(
    "Posibles outliers de cantidad:",
    len(quantity_outliers)
)

"""display(
    quantity_outliers[
        [
            "cd_NumParte",
            "ct_Cantidad",
            "im_PrecioVenta"
        ]
    ].head(20)
)
"""

##Muy importante: OUtlier != Error

Un cliente normalmente compra: 1,4,6,2,4 productos y un día compra 50, esto puede aparecer como outlier, pero quizas fue:
- compra de flotilla
- promoción
- pedido mayorista
- temporada especial

Entonces se deberá considerar como **valor atípico** y no **dato incorrecto**

## Distribución de cantidad (primer gráfica sencilla)

In [0]:
#Gráfica de Data Profiling
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

dfSales["ct_Cantidad"].hist(bins=30)

plt.title("Distribución de cantidad vendida")
plt.xlabel("Cantidad")
plt.ylabel("Frecuencia")

plt.show()



In [0]:
#Boxplot de cantidad
plt.figure(figsize=(10, 4))

plt.boxplot(
    dfSales["ct_Cantidad"].dropna(),
    vert=False
)

plt.title("Distribución de cantidad vendida")
plt.xlabel("Cantidad")

plt.show()

Aquí dropna() si tiene sentido porque no estamos borrando registros del dataset, solo le decimos al gráfico que ignore los valores faltantes.

##Revisar inventario disponible

**Variable Derivada**

In [0]:
#Localizar nombres exactos de existencia y reservado 
dfInventory.columns.to_list()

dfInventory["ct_Disponible"] = (
    dfInventory["ct_Existencia"]
    - dfInventory["ct_Reservado"]
)

display(
    dfInventory[
        [
            "ct_Existencia",
            "ct_Reservado",
            "ct_Disponible"
        ]
    ].describe().T
)

**Buscar disponibilidad negativa**

In [0]:
negative_inventory = dfInventory[
    dfInventory["ct_Disponible"] < 0
]

print(
    "Productos con disponibilidad negativa:",
    len(negative_inventory)
)

display(
    negative_inventory.head(10)
)

**Primera aproximación a movimiento**

Tengo inventario... ¿pero ese inventario realmente se mueve?

In [0]:
#Si el inventario contiene efectivamente la variable de ventas de los últimos 12 meses, podemos hacer:

no_movement = dfInventory[
    dfInventory["ct_VentasUltimos12Meses"] == 0
]

print(
    "Productos sin ventas en los últimos 12 meses:",
    len(no_movement)
)

#Inventario sin movimiento
inventory_no_movement = dfInventory[
    (dfInventory["ct_Existencia"] > 0)
    &
    (dfInventory["ct_VentasUltimos12Meses"] == 0)
]

print(
    "Productos con inventario pero sin movimiento:",
    len(inventory_no_movement)
)



**¿Dónde tengo inventario que no se está moviendo?**

##Observaciones

**Conclusiones de distribución y consistencia**
Después de revisar la distribución de las principales varibales observo que:

1. La cantidad vendida presenta un **rango entre -12.5 y 31.5** unidades.
2. Se identificaron **47 transacciones** con cantidad igual a **cero**.
3. Se identificaron **0** transacciones con **cantidad negativa**.
4. El análisis mediante IQR identificó **0* posibles valores **atípicos**.
5. La existencia disponible presenta **valores entre 20 y -20 **unidades.
6. Se identificaron **461** productos con **inventario disponible negativo**.
7. Se identificaron **265** productos **con existencia pero sin ventas durante los últimos 12 mese**s.


## Ejercicio 

**1. ¿Por qué un outlier no debe eliminarse automáticamente?**

Porque un valor atípico no es necesariamente un dato incorrecto: puede reflejar un comportamiento real del negocio (compra de flotilla, promoción, pedido mayorista, temporada especial). Eliminarlo automáticamente destruiría información legítima. Antes de decidir hay que investigar la causa: si es un error de captura se corrige o elimina con justificación; si es un evento real, se conserva (y en algunos casos hasta se analiza por separado, porque los clientes grandes suelen explicar buena parte de la demanda). Outlier ≠ Error.

**2. ¿Qué diferencia existe entre ct_Existencia y ct_Disponible?**

'ct_Existencia' es el stock físico total registrado en almacén. 'ct_Disponible' es una **variable derivada**: existencia menos lo ya reservado ('ct_Disponible = ct_Existencia - ct_Reservado'). Es decir, es la cantidad que realmente se podría comprometer a nuevos pedidos. La existencia incluye unidades que ya están comprometidas; el disponible es la fracción libre para vender.

**3. ¿Qué significado tendría para el negocio encontrar existencia > 0 y ventas de últimos 12 meses = 0?**

Es **capital de trabajo inmovilizado**: producto que ocupa espacio en almacén, genera costos (almacenaje, seguro, riesgo de obsolescencia) y no genera ingresos. Representa una oportunidad de acción de negocio: revisar si se trata de refacciones obsoletas o descontinuadas, aplicar descuentos o promociones para liquidarlo, redistribuirlo a otra sucursal donde sí rote, o en el peor caso darlo de baja. En este dataset se identificaron 265 productos en esa condición.

**4. Si ct_Disponible < 0, ¿lo corregirías automáticamente a 0? ¿Por qué?**

No, no lo corregiría automáticamente. Un disponible negativo es una **señal de inconsistencia que hay que investigar**, no un dato que se ajusta a ojo. Puede indicar que hay más unidades reservadas que existencias físicas (posible sobreventa, error en el registro de reservas, o existencias mal capturadas). Si lo pongo en 0 sin más, **oculto el síntoma** y el problema de fondo seguirá ocurriendo. Lo correcto es marcarlo como registro para revisión, validar contra el sistema de origen y, una vez identificada la causa, corregir en la fuente o documentar la regla de negocio aplicada. Aquí encontramos 461 productos en esta situación, lo que justifica una revisión sistematizada, no un parche puntual.

**5. ¿Qué indicador te parece más útil para responder “qué productos rotan y cuáles representan una oportunidad”?**

El cruce de **existencia > 0 con ct_VentasUltimos12Meses = 0** (inventario sin movimiento). Este indicador responde directamente a la pregunta de negocio: separa el inventario que rota del que está inmovilizado, cuantifica el capital atrapado y es directamente accionable (liquidar, mover, descontinuar). El disponible negativo, en cambio, es un indicador de calidad del dato, no de rotación. Como paso siguiente, extenderlo a una medida como *meses de cobertura* (existencia / venta promedio mensual) permitiría priorizar no solo los que no se mueven, sino los que se moverán muy lento.

## ¿Cómo se lee el trabajo ralizado hasta este punto?

describe()
    ↓
¿Cómo se comportan generalmente mis datos?

Q1 / Mediana / Q3
    ↓
¿Dónde está concentrada la mayor parte?

IQR = Q3 - Q1
    ↓
¿Cuánto ocupa el 50% central?

Límites IQR
    ↓
¿Qué valores están excepcionalmente alejados?

Outliers
    ↓
¿Por qué ocurrió? La última pregunta ya no la responde Python.
La respondes tú con el contexto de negocio.

## Inventory Distribution
**Objetivo**. Entender cómo se distribuye el inventario e identifciar productos con disponibilidad negativa, exceso aparente o falta de movimiento

In [0]:
#confirmamos los nombres reales 
print("Columnas disponibles en Inventroy:")
for col in dfInventory.columns:
    print(col)

In [0]:
#Seleccionar las variables numéricas relevantes
inventory_numeric_columns = dfInventory.select_dtypes(
    include="number"
).columns.tolist()

print("Columnas numéricas:")
print(inventory_numeric_columns)

In [0]:
#Ejecutar describe
inventory_description = (dfInventory[inventory_numeric_columns].describe())

display(inventory_description)

**Variable Derivada** -> ct_Disponible

In [0]:
#Calculo de la disponibilidad
dfInventory["ct_Disponible"] = (
    dfInventory["ct_Existencia"]
    - dfInventory["ct_Reservado"]
)

display(
    dfInventory[
        ["cd_NumParte", "ct_Existencia", "ct_Reservado", "ct_Disponible"]
    ].head(10)
)

In [0]:
#Describir cd_Disponible
display(
dfInventory["ct_Disponible"].describe()
)

**Análisis de inventario disponible**

El inventario disponible represetna la diferencia entre la existencia registrada y las unidades reservadas:

Disponible = Existencia - Reservado

Esta variable permite aproximar cuántas unidades se encuentran disponibles para etender nuevas ventas

Para interpretar su distribución se revisaron el valor mínimo, los cuartiles, la mediana y el valor máx.

In [0]:
#Detectar disponibilidad negativa
negative_available_inventory = dfInventory[
    dfInventory["ct_Disponible"] < 0
]

print(
    "Productos con disponibilidad negativa:",
    len(negative_available_inventory)
)

display(
    negative_available_inventory[
        [
            "cd_NumParte",
            "ct_Existencia",
            "ct_Reservado",
            "ct_Disponible"
        ]
    ].head(20)
)

tener registros de productos con ct_Disponible en Negativo puede deberse a reglas de negocio como:
- pedidos pendientes
- actualización de invetnario tardía
- regla en DMS
- error de información

Y esta forma de trabajar es consistente con el criterio de calidad se ha venido documentando para Sell Out: una anomalía debe tener regla de validación y tratamiento explícito, no corregirse arbitrariamente aplicando por ejemplo algo como: 
inventory.loc[
    inventory["ct_Disponible"] < 0,
    "ct_Disponible"
] = 0
sin al menos una regla que lo justifique.

In [0]:
#Revisar productos sin existencia
zero_inventory = dfInventory[
    dfInventory["ct_Existencia"] == 0
]

print(
    "Productos con existencia igual a cero:",
    len(zero_inventory)
)

In [0]:
#Identificar inventario sin movimiento
inventory_no_movement = dfInventory[
    (dfInventory["ct_Disponible"] > 0)
    &
    (dfInventory["ct_VentasUltimos12Meses"] == 0)
]

print(
    "Productos con inventario disponible pero sin movimiento:",
    len(inventory_no_movement)
)

display(
    inventory_no_movement.head(20)
)

**Datos importantes a obtener del EDA a invetnaro**

Existencia
     ↓
Reservado
     ↓
Disponible
     ↓
¿Disponible < 0?
     ↓
¿Disponible > 0 y ventas = 0?

1. Mediana de ct_Existencia.
2. Mediana de ct_Disponible.
3. Mínimo de ct_Disponible.
4. Número de productos con ct_Disponible < 0.
5. Número de productos con ct_Disponible > 0 y ventas 12 meses = 0.


Hasta este punto se ha logrado identificar:

datos crudos → validaciones → indicadores interpretables


**Cierre de Inventory Distribution**

Vamos a terminarlo con tres indicadores solamente:
- inventario disponible;
- movimiento;
- cobertura aproximada.

In [0]:
#Visualizar indicadores hast aquí identifciados
display(
inventory_no_movement[
    [
        "cd_NumParte",
        "ct_Existencia",
        "ct_Reservado",
        "ct_Disponible",
        "ct_VentasMesActual",
        "ct_VentasUltimos12Meses"
    ]
].head(20)
)


In [0]:
#inventario disponible
dfInventory["ct_Disponible"] = (
    dfInventory["ct_Existencia"]
    - dfInventory["ct_Reservado"]
)

In [0]:
#Inventario sin movimiento , ¿Qué productos tengo disponibles pero no se están vendiendo?
inventory_no_movement = dfInventory[
    (dfInventory["ct_Disponible"] > 0)
    &
    (dfInventory["ct_VentasUltimos12Meses"] == 0)
]

print(
    "Productos con inventario pero sin movimiento:",
    len(inventory_no_movement)
)

display(
    inventory_no_movement[
        [
            "cd_NumParte",
            "ct_Disponible",
            "ct_VentasUltimos12Meses"
        ]
    ].head(10)
)

**Venta promedio mensual**
Ahora damos un pequeño salto analítico.
Tenemos ventas acumuladas de 12 meses, así que calculamos una aproximación:

VentaPromedioMensual = 12Ventas12Meses / 12

In [0]:
#Venta promedio mensual
dfInventory["ct_VentaPromedioMensual"] = (
    dfInventory["ct_VentasUltimos12Meses"]
    / 12
)

display(
    dfInventory[
        [
            "cd_NumParte",
            "ct_VentasUltimos12Meses",
            "ct_VentaPromedioMensual"
        ]
    ].head(10)
)

In [0]:
#Calcular cobertura
### Al realizar la primera vez esta ejecución se obtuvieron errores por división por cero, por lo que se agregó la condición para evitarlo sería a través del uso de numpy

import numpy as np

dfInventory["ct_CoberturaMeses"] = np.where ( dfInventory["ct_VentaPromedioMensual"] > 0, dfInventory["ct_Disponible"]
    / dfInventory["ct_VentaPromedioMensual"],np.nan
)

# Interpretar el resultado
display(
    dfInventory[
        [
            "cd_NumParte",
            "ct_Disponible",
            "ct_VentaPromedioMensual",
            "ct_CoberturaMeses"
        ]
    ].head(10)
)
#

** Clasificación exploratoria**

In [0]:
def classify_inventory(row):

    if row["ct_Disponible"] > 0 and row["ct_VentasUltimos12Meses"] == 0:
        return "Sin movimiento"

    elif row["ct_CoberturaMeses"] > 6:
        return "Exceso potencial"

    elif row["ct_CoberturaMeses"] < 1:
        return "Riesgo potencial"

    else:
        return "Cobertura adecuada"

In [0]:
dfInventory["tx_ClasificacionInventario"] = dfInventory.apply(
    classify_inventory,
    axis=1
)

In [0]:
#Visualización de cierre
import matplotlib.pyplot as plt

classification_counts = (
    dfInventory["tx_ClasificacionInventario"]
    .value_counts()
)

classification_counts.plot(
    kind="bar",
    figsize=(10,5)
)

plt.title(
    "Clasificación exploratoria de invetnario"
)

plt.xlabel(
    "Clasificación"
)

plt.ylabel(
    "Número de productos"
)

plt.xticks(rotation=45)
plt.show()


##Conclusión del Análisis de Distribución

El análisis permitió observar el comportamiento de las principales variables de ventas e invetnario.

En ventas se revisó la distribución de las cantidades mediante estadística descriptiva, cuartiles e IQR. Los posibles valores atípicos fueron identifciados para revisión, pero no eliminados automáticamente debido a que pueden corresponder a operaciones válidas.

En inventario se calculó la disponibilidad considerando las unidades existentes y reservadas. También se identifciaron productos con inventario pero sin movimiento durante los últimos 12 meses.

Finalmente se calculó una cobertura aproximada utilizando la venta promedio mensual histórica.

A partir de estos indicadores los productos fueron clasificados de forma exploratoria en:
- Sin movimiento
- Exceso potencial
- Riesgo potencial
- Cobertura adecuada

Los umbrales utilizados en esta clasificación corresponden únicamente al prototipo y deberían ser validados posteriormente con las reglas de negocio.

** Resumen de los conceptos utilizados**

- Función describe()
- Concepto y uso de cuartiles;
- Cálculo de IQR;
- Identificando outliers;
- Cálculo disponibilidad;
- Cálculo de productos sin movimiento;
- Cálculo promedio mensual;
- Cálculo de cobertura;
- Clasificación preliminar;
- Visualización.

##¿Podemos integrar correctamente Sales e Inventory usando cd_NumParte?

In [0]:
#Crear conjuntos de productos
sales_parts = set(
    dfSales["cd_NumParte"].dropna()
)

inventory_parts = set(
    dfInventory["cd_NumParte"].dropna()
)

In [0]:
#Productos en común
commun_parts = (
    sales_parts & inventory_parts
)

print(
    "Productos presentes en ambos datasets", len(commun_parts)
)

In [0]:
#Productos que existen solamente en Ventas
sales_only =(
    sales_parts - inventory_parts
)

print(
    "Productos presentes en Ventas, pero no en Inventario", len(sales_only)
)
#Productos que existen solamente en Inventario
inventory_only =(
    inventory_parts - sales_parts
)

print(
    "Productos presentes en Inventario, pero no en Ventas",len(inventory_only)
)
